In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
#os.chdir('c:/py/data')
os.chdir('/content/drive/MyDrive/repo/Lexi') #<-- your local
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/repo/Lexi


In [5]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

def get_eagle_farm_files():
    """Get all Eagle Farm form and results files, sorted by date in filename"""
    base_dir = 'research/2026/01/'

    # Find all form and results files
    form_files = []
    results_files = []

    for filename in os.listdir(base_dir):
        if 'form-data' in filename and 'eagle-farm' in filename:
            form_files.append(filename)
        elif 'results-data' in filename and 'eagle-farm' in filename:
            results_files.append(filename)

    # Sort files by date in filename (extract date from filename)
    def extract_date_from_filename(filename):
        # Extract date pattern like 2026-01-07 or 260107
        date_match = re.search(r'(\d{4}-\d{2}-\d{2})|(\d{6})', filename)
        if date_match:
            date_str = date_match.group(0)
            if '-' in date_str:
                # Format: YYYY-MM-DD
                return date_str
            else:
                # Format: YYMMDD
                year = '20' + date_str[:2]
                month = date_str[2:4]
                day = date_str[4:6]
                return f"{year}-{month}-{day}"
        return "0000-00-00"  # Default for files without dates

    form_files.sort(key=extract_date_from_filename)
    results_files.sort(key=extract_date_from_filename)

    return form_files, results_files

def load_and_concatenate_data():
    """Load and concatenate all Eagle Farm form and results data in date order"""
    form_files, results_files = get_eagle_farm_files()

    print(f"Found {len(form_files)} form files: {form_files}")
    print(f"Found {len(results_files)} results files: {results_files}")

    # Load and concatenate form data
    form_data_list = []
    for form_file in form_files:
        filepath = f'research/2026/01/{form_file}'
        try:
            df = pd.read_csv(filepath)
            print(f"Loaded form data from {form_file} with shape: {df.shape}")
            form_data_list.append(df)
        except FileNotFoundError:
            print(f"Warning: Form file {filepath} not found")

    # Load and concatenate results data
    results_data_list = []
    for results_file in results_files:
        filepath = f'research/2026/01/{results_file}'
        try:
            df = pd.read_csv(filepath)
            print(f"Loaded results data from {results_file} with shape: {df.shape}")
            results_data_list.append(df)
        except FileNotFoundError:
            print(f"Warning: Results file {filepath} not found")

    if form_data_list:
        all_form_data = pd.concat(form_data_list, ignore_index=True)
        print(f"Combined form data shape: {all_form_data.shape}")
    else:
        print("No form data files found. Creating sample data...")
        # Create sample data with similar structure to the actual form data
        n_samples = 100
        all_form_data = pd.DataFrame({
            'Trainer.FullName': [f'Trainer_{i}' for i in range(n_samples)],
            'Jockey.FullName': [f'Jockey_{i}' for i in range(n_samples)],
            'Name': [f'Horse_{i}' for i in range(n_samples)],
            'Age': np.random.randint(2, 10, n_samples),
            'Sex': np.random.choice(['Horse', 'Gelding', 'Mare'], n_samples),
            'Weight': np.random.uniform(45, 65, n_samples),
            'Barrier': np.random.randint(1, 15, n_samples),
            'TabNo': np.random.randint(1, 20, n_samples),
            'PrizeMoney': np.random.exponential(100000, n_samples),
            'CareerStarts': np.random.poisson(20, n_samples),
            'CareerWins': np.random.poisson(3, n_samples),
            'CareerSeconds': np.random.poisson(2, n_samples),
            'CareerThirds': np.random.poisson(3, n_samples),
            'WinPct': np.random.beta(2, 5, n_samples),
            'PlacePct': np.random.beta(3, 4, n_samples),
            'GoodRecord.Firsts': np.random.poisson(2, n_samples),
            'GoodRecord.Seconds': np.random.poisson(1, n_samples),
            'GoodRecord.Thirds': np.random.poisson(1, n_samples),
            'DistanceRecord.Firsts': np.random.poisson(1, n_samples),
            'TrackRecord.Firsts': np.random.poisson(1, n_samples),
        })

    if results_data_list:
        all_results_data = pd.concat(results_data_list, ignore_index=True)
        print(f"Combined results data shape: {all_results_data.shape}")
    else:
        print("No results data files found. Creating sample data...")
        # Create sample results data with similar structure to the actual results data
        n_samples = 100
        all_results_data = pd.DataFrame({
            'horse': [f'Horse_{i}' for i in range(n_samples)],
            'position': np.random.randint(1, 15, n_samples),
            'margin': np.random.exponential(3, n_samples),
            'time': [f"{np.random.randint(50, 120)}" for _ in range(n_samples)],
            'track': ['Eagle Farm'] * n_samples,
            'race number': np.random.randint(1, 10, n_samples),
            'price sp': np.random.uniform(1.5, 50, n_samples)
        })

    return all_form_data, all_results_data

def merge_and_engineer_features(form_data, results_data):
    """Merge form and results data and create engineered features"""
    # Normalize horse names for merging
    form_data['horse_normalized'] = form_data['Name'].str.lower().str.replace(' ', '').str.replace("'", "")
    results_data['horse_normalized'] = results_data['horse'].str.lower().str.replace(' ', '').str.replace("'", "")

    # Merge datasets
    merged_data = pd.merge(
        form_data,
        results_data[['horse_normalized', 'position', 'race number', 'margin', 'price sp']],
        on='horse_normalized',
        how='inner'
    )

    print(f"Merged dataset shape: {merged_data.shape}")
    print(f"Unique races: {merged_data['race number'].nunique()}")
    print(f"Positions range: {merged_data['position'].min()} to {merged_data['position'].max()}")

    # Feature engineering
    if 'WinPct' in merged_data.columns:
        merged_data['combined_skill'] = merged_data['WinPct'] * merged_data['PlacePct']
    else:
        merged_data['combined_skill'] = np.random.beta(2, 3, len(merged_data))

    if 'GoodRecord.Firsts' in merged_data.columns:
        merged_data['experience_score'] = (
            merged_data['GoodRecord.Firsts'] +
            merged_data['GoodRecord.Seconds'] +
            merged_data['GoodRecord.Thirds']
        ) / 3
    else:
        merged_data['experience_score'] = np.random.poisson(2, len(merged_data))

    if 'Weight' in merged_data.columns and 'DistanceRecord.Firsts' in merged_data.columns:
        merged_data['form_efficiency'] = merged_data['DistanceRecord.Firsts'] / (merged_data['Weight'] + 1)
    else:
        merged_data['form_efficiency'] = np.random.normal(0.1, 0.05, len(merged_data))

    if 'Barrier' in merged_data.columns:
        merged_data['barrier_advantage'] = 8.5 - abs(merged_data['Barrier'] - 8.5)
    else:
        merged_data['barrier_advantage'] = np.random.uniform(-3, 3, len(merged_data))

    # Encode categorical variables
    categorical_cols = ['Trainer.FullName', 'Jockey.FullName', 'Sex']
    for col in categorical_cols:
        if col in merged_data.columns:
            le = LabelEncoder()
            # Handle unknown categories by fitting on all data first
            le.fit(merged_data[col].astype(str))
            merged_data[f'{col}_encoded'] = le.transform(merged_data[col].astype(str))

    return merged_data

def prepare_features(merged_data):
    """Select and prepare features for modeling based on available columns"""
    # Define potential feature columns based on typical form data
    potential_features = [
        'Age', 'Weight', 'Barrier', 'TabNo', 'PrizeMoney',
        'CareerStarts', 'CareerWins', 'CareerSeconds', 'CareerThirds',
        'WinPct', 'PlacePct', 'combined_skill', 'experience_score',
        'form_efficiency', 'barrier_advantage', 'price sp',
        'Trainer.FullName_encoded', 'Jockey.FullName_encoded', 'Sex_encoded',
        'GoodRecord.Firsts', 'GoodRecord.Seconds', 'GoodRecord.Thirds',
        'DistanceRecord.Firsts', 'TrackRecord.Firsts'
    ]

    # Filter to only available columns
    feature_columns = [col for col in potential_features if col in merged_data.columns]

    # Limit to first 15 features to avoid overfitting if too many are available
    #feature_columns = feature_columns[:15] if len(feature_columns) > 15 else feature_columns

    print(f"Selected {len(feature_columns)} features for modeling: {feature_columns}")

    X = merged_data[feature_columns]
    y_position = merged_data['position']

    return X, y_position, feature_columns

def preprocess_data(X):
    """Handle missing values and standardize features"""
    # Handle missing values
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    # Standardize features
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X_imputed.columns)

    return X_scaled, imputer, scaler

def train_regression_model(X_train, y_train):
    """Train Gradient Boosting model for regression"""
    gb_regressor = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42
    )
    gb_regressor.fit(X_train, y_train)
    return gb_regressor

def evaluate_regression_model(model, X_test, y_test):
    """Evaluate regression model and return metrics"""
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'predictions': y_pred
    }

def k_fold_cross_validation(X, y, n_splits=5):
    """Perform k-fold cross-validation for regression"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Define the model
    model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42
    )

    # Perform cross-validation for different metrics
    mse_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')
    mae_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    r2_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')

    # Convert negative MSE to positive
    mse_scores = -mse_scores

    return {
        'mse': mse_scores,
        'mae': -mae_scores,  # Negative MAE needs to be converted back
        'r2': r2_scores
    }

def analyze_feature_importance(reg_model, feature_columns):
    """Analyze and return feature importances"""
    reg_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': reg_model.feature_importances_
    }).sort_values('importance', ascending=False)

    return reg_importance

def generate_detailed_report(model, X, y, merged_data, feature_columns):
    """Generate detailed report with predicted and actual positions for all horses"""
    # Make predictions on the full dataset
    all_pos_predictions = model.predict(X)

    # Create detailed report with predicted positions rounded to 0 decimal places
    detailed_report = pd.DataFrame({
        'Horse_Name': merged_data.get('Name', [f'Horse_{i}' for i in range(len(merged_data))]),
        'Actual_Position': y.values,
        'Predicted_Position': np.round(all_pos_predictions, 0).astype(int),
        'Position_Error': np.abs(y.values - np.round(all_pos_predictions, 0).astype(int))
    })

    return detailed_report

def main():
    print("FORM-TO-RESULTS REGRESSION ANALYSIS WITH GRADIENT BOOSTING")
    print("Loading and concatenating all Eagle Farm form and results data in date order...")
    print("="*60)

    # Step 1: Load and concatenate all Eagle Farm data in date order
    print("Step 1: Loading and concatenating all Eagle Farm form and results data...")
    form_data, results_data = load_and_concatenate_data()
    print(f"Form data shape: {form_data.shape}")
    print(f"Results data shape: {results_data.shape}")

    # Step 2: Merge and engineer features
    print("\nStep 2: Merging data and engineering features...")
    merged_data = merge_and_engineer_features(form_data, results_data)
    print(f"Merged data shape: {merged_data.shape}")

    # Step 3: Prepare features
    print("\nStep 3: Preparing features for modeling...")
    X, y_position, feature_columns = prepare_features(merged_data)
    print(f"Feature matrix shape: {X.shape}")
    print(f"Target variable shape: {y_position.shape}")

    # Step 4: Preprocess data
    print("\nStep 4: Preprocessing data...")
    X_scaled, imputer, scaler = preprocess_data(X)
    print("Data preprocessing completed")

    # Step 5: Split data
    print("\nStep 5: Splitting data into train/test sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_position, test_size=0.2, random_state=42
    )
    print(f"Training set shape: {X_train.shape}")
    print(f"Test set shape: {X_test.shape}")

    # Step 6: Train regression model
    print("\nStep 6: Training Gradient Boosting regression model...")
    gb_regressor = train_regression_model(X_train, y_train)
    print("Regression model trained successfully")

    # Step 7: Evaluate model on test set
    print("\nStep 7: Evaluating model on test set...")
    reg_metrics = evaluate_regression_model(gb_regressor, X_test, y_test)
    print("REGRESSION METRICS (Test Set):")
    print(f"  MSE: {reg_metrics['mse']:.3f}")
    print(f"  RMSE: {reg_metrics['rmse']:.3f}")
    print(f"  MAE: {reg_metrics['mae']:.3f}")
    print(f"  R²: {reg_metrics['r2']:.3f}")

    # Step 8: Perform 5-fold cross-validation
    print("\nStep 8: Performing 5-fold cross-validation...")
    cv_results = k_fold_cross_validation(X_scaled, y_position, n_splits=5)

    print("5-FOLD CROSS-VALIDATION RESULTS:")
    print(f"  MSE: {cv_results['mse'].mean():.3f} (+/- {cv_results['mse'].std() * 2:.3f})")
    print(f"  RMSE: {np.sqrt(cv_results['mse']).mean():.3f} (+/- {np.sqrt(cv_results['mse']).std() * 2:.3f})")
    print(f"  MAE: {cv_results['mae'].mean():.3f} (+/- {cv_results['mae'].std() * 2:.3f})")
    print(f"  R²: {cv_results['r2'].mean():.3f} (+/- {cv_results['r2'].std() * 2:.3f})")

    # Step 9: Feature importance analysis
    print("\nStep 9: Analyzing feature importance...")
    reg_importance = analyze_feature_importance(gb_regressor, feature_columns)
    print("TOP 10 FEATURES - REGRESSION (Position Prediction):")
    print(reg_importance.head(10))

    # Step 10: Generate detailed report for all horses
    print("\nStep 10: Generating detailed prediction report for all horses in the dataset...")
    detailed_report = generate_detailed_report(gb_regressor, X_scaled, y_position, merged_data, feature_columns)
    print("\nDETAILED PREDICTION REPORT FOR ALL HORSES:")
    print(detailed_report.to_string(index=False))

    # Show top 10 largest prediction errors
    print(f"\nTOP 10 LARGEST POSITION PREDICTION ERRORS:")
    top_errors = detailed_report.nlargest(10, 'Position_Error')
    print(top_errors[['Horse_Name', 'Actual_Position', 'Predicted_Position', 'Position_Error']].to_string(index=False))

    # Step 11: Summary
    print("\nFORM-TO-RESULTS REGRESSION ANALYSIS SUMMARY")
    print("="*50)
    print(f"Data Shape: {merged_data.shape}")
    print(f"Features Used: {len(feature_columns)}")
    print()
    print("REGRESSION PERFORMANCE (Test Set):")
    print(f"  - MSE: {reg_metrics['mse']:.3f}")
    print(f"  - RMSE: {reg_metrics['rmse']:.3f}")
    print(f"  - MAE: {reg_metrics['mae']:.3f}")
    print(f"  - R²: {reg_metrics['r2']:.3f}")
    print()
    print("5-FOLD CROSS-VALIDATION PERFORMANCE:")
    print(f"  - Avg MSE: {cv_results['mse'].mean():.3f}")
    print(f"  - Avg RMSE: {np.sqrt(cv_results['mse']).mean():.3f}")
    print(f"  - Avg MAE: {cv_results['mae'].mean():.3f}")
    print(f"  - Avg R²: {cv_results['r2'].mean():.3f}")
    print()
    print("KEY FINDINGS:")
    print(f"  - Top feature for position prediction: {reg_importance.iloc[0]['feature']}")
    print(f"  - Model explains {(reg_metrics['r2']*100):.1f}% of variance in finishing positions")
    print()
    print("USAGE INSTRUCTIONS:")
    print("  1. Prepare new race data with the same features")
    print("  2. Apply the same preprocessing steps (imputation, scaling)")
    print("  3. Use the trained model to predict finishing positions")
    print("  4. Use the detailed report to analyze individual horse predictions")

if __name__ == "__main__":
    main()

FORM-TO-RESULTS REGRESSION ANALYSIS WITH GRADIENT BOOSTING
Loading and concatenating all Eagle Farm form and results data in date order...
Step 1: Loading and concatenating all Eagle Farm form and results data...
Found 2 form files: ['form-data-2026-01-28-eagle-farm.csv', 'form-data-2026-01-31-eagle-farm.csv']
Found 2 results files: ['results-data-2026-01-07-eagle-farm.csv', 'results-data-2026-01-28-eagle-farm.csv']
Loaded form data from form-data-2026-01-28-eagle-farm.csv with shape: (71, 781)
Loaded form data from form-data-2026-01-31-eagle-farm.csv with shape: (118, 781)
Loaded results data from results-data-2026-01-07-eagle-farm.csv with shape: (81, 23)
Loaded results data from results-data-2026-01-28-eagle-farm.csv with shape: (71, 23)
Combined form data shape: (189, 781)
Combined results data shape: (152, 23)
Form data shape: (189, 781)
Results data shape: (152, 23)

Step 2: Merging data and engineering features...
Merged dataset shape: (87, 786)
Unique races: 8
Positions range: 